In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp
import sys

In [2]:
spark = SparkSession.builder \
        .appName("FAOSTAT Data Ingestion") \
        .config("spark.executor.memory", "4g") \
        .config("spark.driver.memory", "2g") \
        .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
        .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
        .config("spark.sql.catalog.lakehouse.catalog-impl", "org.apache.iceberg.jdbc.JdbcCatalog") \
        .config("spark.sql.catalog.lakehouse.uri", "jdbc:postgresql://localhost:5432/metastore") \
        .config("spark.sql.catalog.lakehouse.jdbc.user", "admin") \
        .config("spark.sql.catalog.lakehouse.jdbc.password", "password") \
        .config("spark.sql.catalog.lakehouse.warehouse", "s3a://lakehouse/warehouse") \
        .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
        .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
        .config("spark.hadoop.fs.s3a.path.style.access", "true") \
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
        .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.hadoop:hadoop-aws:3.3.4,org.postgresql:postgresql:42.6.0,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
        .getOrCreate()

:: loading settings :: url = jar:file:/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lap16470/.ivy2/cache
The jars for the packages stored in: /Users/lap16470/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
org.postgresql#postgresql added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cc79dd15-2b82-4882-aa04-547b215c4469;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.4.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.postgresql#postgresql;42.6.0 in local-m2-cache
	found org.checkerframework#checker-qual;3.31.0 in local-m2-cache
:: resolution report :: resolve 1382ms :: artifacts dl 19ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	o

26/03/14 16:03:50 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [3]:
spark.sql("show tables from lakehouse.bronze").show()

+---------+---------------+-----------+
|namespace|      tableName|isTemporary|
+---------+---------------+-----------+
|   bronze|faostat_qcl_raw|      false|
|   bronze|faostat_tcl_raw|      false|
+---------+---------------+-----------+



In [4]:
spark.sql("select * from lakehouse.bronze.faostat_qcl_raw limit 1").show()

26/03/14 14:30:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/03/14 14:30:50 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+
|Area Code|Area Code (M49)|       Area|Item Code|Item Code (CPC)|             Item|Element Code|       Element|Year Code|Year|Unit|Value|Flag|Note|      ingestion_time|
+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+
|        2|           '004|Afghanistan|      221|         '01371|Almonds, in shell|        5312|Area harvested|     1961|1961|  ha|  0.0|   A|NULL|2026-03-14 14:26:...|
+---------+---------------+-----------+---------+---------------+-----------------+------------+--------------+---------+----+----+-----+----+----+--------------------+



In [15]:
spark.sql(""" 
    SELECT COUNT(DISTINCT Item)
    FROM lakehouse.bronze.faostat_qcl_raw
    WHERE Element = 'Production'
""").show(truncate=False, vertical=True)

-RECORD 0-------------------
 count(DISTINCT Item) | 280 



In [38]:
spark.sql(""" 
    SELECT DISTINCT Area
    FROM lakehouse.bronze.faostat_qcl_raw
    WHERE Element = 'Production'
    ORDER BY Area
""").show(truncate=False, vertical=True)

-RECORD 0-------------------------
 Area | Afghanistan               
-RECORD 1-------------------------
 Area | Africa                    
-RECORD 2-------------------------
 Area | Albania                   
-RECORD 3-------------------------
 Area | Algeria                   
-RECORD 4-------------------------
 Area | Americas                  
-RECORD 5-------------------------
 Area | Angola                    
-RECORD 6-------------------------
 Area | Antigua and Barbuda       
-RECORD 7-------------------------
 Area | Argentina                 
-RECORD 8-------------------------
 Area | Armenia                   
-RECORD 9-------------------------
 Area | Asia                      
-RECORD 10------------------------
 Area | Australia                 
-RECORD 11------------------------
 Area | Australia and New Zealand 
-RECORD 12------------------------
 Area | Austria                   
-RECORD 13------------------------
 Area | Azerbaijan                
-RECORD 14----------

In [22]:
spark.sql(""" 
    SELECT Unit, count(*)
    FROM lakehouse.bronze.faostat_qcl_raw
    WHERE Element = 'Production'
    GROUP BY Unit
""").show(truncate=False, vertical=True)

-RECORD 0-----------
 Unit     | 1000 No 
 count(1) | 18208   
-RECORD 1-----------
 Unit     | t       
 count(1) | 1625403 



In [23]:
raw_df = spark.read.table("lakehouse.bronze.faostat_qcl_raw")

In [26]:
production_df = raw_df.filter(col("Element") == "Production") \
    .select(
        col("Area").alias("country"),
        col("Item").alias("crop"),
        col("Year").cast("int").alias("year"),
        col("Value").cast("double").alias("production_tonnes"),
        col("Unit").alias("unit")
    )

In [33]:
production_df.printSchema()

root
 |-- country: string (nullable = true)
 |-- crop: string (nullable = true)
 |-- year: integer (nullable = true)
 |-- production_tonnes: double (nullable = true)
 |-- unit: string (nullable = true)



In [39]:
from pyspark.sql.functions import sum
yearly_global = production_df.groupBy("year", "crop").agg(sum("production_tonnes").alias("global_production"))

In [40]:
yearly_global.show(10, truncate=False)

+----+---------------------------------------------+--------------------+
|year|crop                                         |global_production   |
+----+---------------------------------------------+--------------------+
|1982|Almonds, in shell                            |4863026.0           |
|1996|Almonds, in shell                            |5852574.77          |
|1967|Butter and ghee of sheep milk                |250147.54           |
|1979|Butter of cow milk                           |2.66341191E7        |
|2021|Cattle fat, unrendered                       |1.3907227889999997E7|
|1961|Cheese from milk of sheep, fresh or processed|1916286.6500000001  |
|2021|Cheese from milk of sheep, fresh or processed|2714991.5300000003  |
|2005|Grapes                                       |3.0755777946E8      |
|1997|Hen eggs in shell, fresh                     |4.16664672649E9     |
|2014|Meat of camels, fresh or chilled             |3046765.5700000003  |
+----+--------------------------------

In [42]:
top_crops = production_df.groupBy("crop").agg(sum("production_tonnes").alias("global_production")) \
    .orderBy(col("global_production").desc()) \
    .limit(20)

In [44]:
top_crops.show(truncate=False)

+------------------------------+---------------------+
|crop                          |global_production    |
+------------------------------+---------------------+
|Cereals, primary              |5.718534700478298E11 |
|Sugar Crops Primary           |3.9754631556152E11   |
|Sugar cane                    |3.2315912648324994E11|
|Hen eggs in shell, fresh      |2.5165575238421005E11|
|Roots and Tubers, Total       |1.9987241223368002E11|
|Vegetables Primary            |1.7472151587100983E11|
|Maize (corn)                  |1.7174773425859003E11|
|Milk, Total                   |1.6661925099221997E11|
|Rice                          |1.5592062489107983E11|
|Wheat                         |1.5315627567287012E11|
|Fruit Primary                 |1.5261546497652023E11|
|Raw milk of cattle            |1.4223613403094006E11|
|Potatoes                      |9.011076274889001E10 |
|Sugar beet                    |7.409734343813E10    |
|Meat, Total                   |5.9362016138129944E10|
|Cassava, 

In [4]:
raw_df = spark.read.table("lakehouse.bronze.faostat_tcl_raw")

In [47]:
spark.sql("""
    SELECT *
    FROM lakehouse.bronze.faostat_tcl_raw
    LIMIT 1
""").show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------
 Area Code       | 2                                              
 Area Code (M49) | '004                                           
 Area            | Afghanistan                                    
 Item Code       | 221                                            
 Item Code (CPC) | '01371                                         
 Item            | Almonds, in shell                              
 Element Code    | 5610                                           
 Element         | Import quantity                                
 Year Code       | 2014                                           
 Year            | 2014                                           
 Unit            | t                                              
 Value           | 34.46                                          
 Flag            | X                                              
 Note            | Estimated data using trading partners datab

In [52]:
spark.sql("""
    SELECT *
    FROM lakehouse.bronze.faostat_tcl_raw
    WHERE Element = 'Import value'
    LIMIT 1
""").show(truncate=False, vertical=True)

-RECORD 0---------------------------------------------------------
 Area Code       | 2                                              
 Area Code (M49) | '004                                           
 Area            | Afghanistan                                    
 Item Code       | 221                                            
 Item Code (CPC) | '01371                                         
 Item            | Almonds, in shell                              
 Element Code    | 5622                                           
 Element         | Import value                                   
 Year Code       | 2014                                           
 Year            | 2014                                           
 Unit            | 1000 USD                                       
 Value           | 156.0                                          
 Flag            | X                                              
 Note            | Estimated data using trading partners datab

In [5]:
trade_df = raw_df.select(
    col("Area").alias("country"),
    col("Item").alias("product"),
    col("Year").cast("int").alias("year"),
    col("Value").cast("double").alias("trade_value"),
    col("Unit").alias("unit"),
    col("Element").alias("trade_type")
).filter(col("trade_value").isNotNull())

In [6]:
trade_df.show(1)

+-----------+-----------------+----+-----------+----+---------------+
|    country|          product|year|trade_value|unit|     trade_type|
+-----------+-----------------+----+-----------+----+---------------+
|Afghanistan|Almonds, in shell|2014|      34.46|   t|Import quantity|
+-----------+-----------------+----+-----------+----+---------------+
only showing top 1 row



In [7]:
from pyspark.sql.functions import when
trade_balance_df = trade_df.groupBy("country", "year", "product") \
    .pivot("trade_type", ["Import quantity", "Export quantity"]) \
    .sum("trade_value") \
    .withColumnRenamed("Import quantity", "imports") \
    .withColumnRenamed("Export quantity", "exports") \
    .fillna(0) \
    .withColumn("trade_balance", col("exports") - col("imports")) \
    .withColumn("net_position", 
                when(col("trade_balance") > 0, "Net Exporter")
                .when(col("trade_balance") < 0, "Net Importer")
                .otherwise("Balanced"))

In [8]:
trade_balance_df.show(2)

26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/03/14 15:36:05 WARN RowBasedKeyValueBatch: Calling spill() on

+-----------+----+--------------------+-------+-------+-------------+------------+
|    country|year|             product|imports|exports|trade_balance|net_position|
+-----------+----+--------------------+-------+-------+-------------+------------+
|Afghanistan|1965|Cereal preparatio...| 1700.0|    0.0|      -1700.0|Net Importer|
|Afghanistan|1966|           Olive oil|    1.0|    0.0|         -1.0|Net Importer|
+-----------+----+--------------------+-------+-------+-------------+------------+
only showing top 2 rows



In [7]:
spark.sql("show tables from lakehouse.gold").show(truncate=False, vertical=True)

-RECORD 0-----------------------------
 namespace   | gold                   
 tableName   | top_crops              
 isTemporary | false                  
-RECORD 1-----------------------------
 namespace   | gold                   
 tableName   | trade_balance          
 isTemporary | false                  
-RECORD 2-----------------------------
 namespace   | gold                   
 tableName   | yearly_crop_production 
 isTemporary | false                  



In [8]:
spark.sql("select * from lakehouse.gold.trade_balance limit 1").show(truncate=False, vertical=True)

-RECORD 0---------------------------
 country       | Afghanistan        
 year          | 1961               
 product       | Butter of cow milk 
 imports       | 23.0               
 exports       | 0.0                
 trade_balance | -23.0              
 net_position  | Net Importer       

